playground is to work on current issues and testing new functionalities with newest version of 2d_combined_playground.py

current issues:
- traditional spec image looks really weird when testing in .py file
- rendering breaks when trying to use new_utils.axis_labels() in non_rgb_iteration()

new functionality:
- adding in non-rgb line graph code

In [40]:
import pandas as pd
import matplotlib.pyplot as plt

import matplotlib.ticker as ticker  # noqa: PLR0402

import io
import re
import nist_codes
import numpy as np


import os
import random
import csv

import plot_funcs

from scipy.signal import find_peaks

In [39]:
# Shared Values
Y_TITLE = 'Intensity'
X_TITLE = 'Wavelength (nm)'
COLOUR = 'white'
BG = 'black'
X_MIN = 400
X_MAX = 750
FIG_WIDTH = 15
DPI = 600

# Traditional Plot Values
FIG_HEIGHT_BASE = 3.0
FIG_SIZE = (FIG_WIDTH, FIG_HEIGHT_BASE)
MIN_NEEDLE_WIDTH = 0.1
MAX_NEEDLE_WIDTH = 0.3
MAX_Y_SCALE = 0.75
NEEDLE_POWER_SHAPE = 4
LABEL_NORM_INT = 0.20
GLOW_WIDTH_MULT = 1.3

# Dynamic Height Values (overflow section)
fig_height_overflow_scale = 9.0

# Normalised-Specific Values
NORM_PROM_PERC = 0.15
NORM_MIN_BRIGHT = 0.01
NORM_GLOW_ALPHA = 0
NORM_PEAK_EMPHASIS = 1.1
NORM_PEAK_LABEL_POSN = 0.75

# "Default" Values (i.e. for not normalised)
DEFAULT_PROM_PERC = 0.08    # also used by "other" plots
DEFAULT_MIN_BRIGHT = 0.1    # also used by "other" plots
DEFAULT_GLOW_ALPHA = 0.35
DEFAULT_PEAK_EMPHASIS = 1.4
DEFAULT_PEAK_LABEL_POSN = 0.77

# Other Plot Values
fig_size = (15,6)
#prominence = 0.08       # changed from 0.12
#min_bright = 0.1
min_alpha = 0.1
min_alpha_scatter = 0.2
base_marker_size = 5
max_marker_size_factor = 95
gamma_factor = 0.8
bar_width = 1
smoothing_window = 5    # Increase this value to control the degree of smoothing
base_sigma_nm = 0.5 # Base width of the Gaussian (for low intensity peaks)
max_sigma_multiplier = 4.0 # How much wider the highest intensity peaks can be
reverse_x = True
plot_type = None
show_grid = True


major_locator = ticker.MultipleLocator(50)
minor_locator = ticker.MultipleLocator(10)


lambda_tokens = ["nm", "wavelength", "wavelength_nm", "lambda", "lambda_nm", "wl", "wl_nm", "Observed", "Observed Wavelength", "obs", "wave", "w"]

int_tokens =["Grey Val", "grey val", "gray val", "grayscale", "gray value", "intensity", "signal", "counts", "value", "int", "rel. int.", "grey", "Rel. Int.", "Relative Intensity", "Rel Int", "Intensity", "A", "Aki", "gA", "gf", "weighted f", "f", "Intensity/Counts", 'rel', 'count', 'flux', 'grey value', 'i']


In [41]:
def nist_check(data_df, detect_columns, nm_col, int_col, force_nist=None):
    df_data = data_df.copy()
    res_col_names(data_df=data_df, detect_columns=detect_columns, nm_col=nm_col, int_col=int_col,)
    df_data[wl_col] = pd.to_numeric(df_data[wl_col], errors='coerce')
    has_any_nist, nist_diag = nist_codes.detect_nist_values(df_data[INT_col], force_nist=force_nist)
    if has_any_nist:
        has_nist = True
    else:
        has_nist = False
    return has_nist, nm_col, int_col

def run_nist_check(data_df, detect_columns, nm_col, int_col, force_nist=None):
    res_col_names(data_df=data_df, detect_columns=detect_columns, nm_col=nm_col, int_col=int_col,)
    df_data = data_df.copy()
    df_data[wl_col] = pd.to_numeric(df_data[wl_col], errors='coerce')
    has_any_nist, nist_diag = nist_codes.detect_nist_values(df_data[INT_col], force_nist=force_nist)
    return df_data, has_any_nist, detect_columns, nm_col, int_col


def prepare_generic_spectrum(df, int_col, apply_descriptor_adjustments=False):
    df = df.copy()
    df['_raw_int'] = pd.to_numeric(df[int_col], errors='coerce')
    df['_descriptor'] = ''
    df['_intensity_mult'] = 1.0
    df['_width_mult'] = 1.0
    df['_include'] = True
    df['_adj_int'] = df['_raw_int']
    return df


def prep_with_nist(data_df, detect_columns, nm_col, int_col, x_min = X_MIN, x_max = X_MAX, apply_descriptor_adjustments = False):
    global int_range

    df_plot_data, has_any_nist = run_nist_check(data_df=data_df, detect_columns=detect_columns, nm_col=nm_col, int_col=int_col,)
    if has_any_nist:
        df_plot_data = nist_codes.prepare_nist_spectrum(df_plot_data, INT_col, apply_descriptor_adjustments)
        print('NIST Destriptors Detected. Preparing NIST Rendering.')
    else:
        df_plot_data = prepare_generic_spectrum(df_plot_data, INT_col, apply_descriptor_adjustments)
        print('No NIST Destriptors Detected. Preparing Generic Rendering.')

    # parse NIST-style cells
    parsed_intensity = df_plot_data[INT_col].apply(nist_codes.parse_nist_intensity)
    df_plot_data['_raw_int'] = parsed_intensity.apply(lambda t: t[0])
    df_plot_data['_descriptor'] = parsed_intensity.apply(lambda t: t[1])

    # drop rows missing wavelength or numeric intensity
    df_plot_data = df_plot_data.dropna(subset=[wl_col, '_raw_int']).copy()
    df_plot_data = df_plot_data[(df_plot_data[wl_col] >= x_min) & (df_plot_data[wl_col] <= x_max)].copy()
    df_plot_data = df_plot_data.sort_values(by=wl_col).reset_index(drop=True)

    # returns empty flag for empty datasets
    if df_plot_data.empty:
        return None, True 

    # Normalize using adjusted intensity
    min_int_val = df_plot_data['_adj_int'].min()
    max_int_val = df_plot_data['_adj_int'].max()
    int_range = max_int_val - min_int_val
    
    if int_range == 0 or np.isnan(int_range):
        df_plot_data['Norm_Int'] = 1.0
    else:
        df_plot_data['Norm_Int'] = (df_plot_data['_adj_int'] - min_int_val) / int_range

    return df_plot_data, False, has_any_nist



def prep_other(data_df, detect_columns, nm_col, int_col, x_min = X_MIN, x_max = X_MAX):
    global int_range
    global y_max

    res_col_names(data_df=data_df, detect_columns=detect_columns, nm_col=nm_col, int_col=int_col,)
    df_filtered = data_df.copy()
    df_filtered = df_filtered[(df_filtered[wl_col] >= x_min) & (df_filtered[wl_col] <= x_max)].copy()
    df_filtered = df_filtered.sort_values(by=wl_col).reset_index(drop=True)

    min_int = df_filtered[INT_col].min()
    max_int = df_filtered[INT_col].max()

    int_range = max_int - min_int
    int_vals = df_filtered[INT_col]
    y_max = int_vals.max()
    print('(prep_other) ymax = ', y_max)

    if int_range == 0:
        df_filtered['Norm_Int'] = 1.0
    else:
        df_filtered['Norm_Int'] = (df_filtered[INT_col] - min_int) / int_range

    if df_filtered.empty:
        return None, True 

    return df_filtered, False





def get_rgb_type(graph_type):
    global rgb_type
    if graph_type != 'non rgb line':
        rgb_type = 'yes'
    else:
        rgb_type = 'no'



def axis_labels(
    graph_type,
    show_grid,
    show_peak_labels,
    title,
    random_title,
    save_path=None,
    dpi = DPI,
    fig_size = fig_size,
    reverse_x = reverse_x,
    x_min = X_MIN,
    x_max = X_MAX,
    x_title = X_TITLE,
    y_title = Y_TITLE,
    y_min = 0,
    y_max = 0,
    fig_bg = BG,
    text = COLOUR,
):
    
    plt.figure(figsize=fig_size)
    ax = plt.gca()

    plt.gcf().set_facecolor(fig_bg)
    for spine in ax.spines.values():        # new block. may have to take out or decrease linewidth. 
        spine.set_linewidth(0.3)
        spine.set_color('darkgrey')
    if show_grid is True:
        plt.grid(True, color='darkgrey', linewidth=0.25)   # took out 'linestyle = ':' '    may have to add back in. 
    elif show_grid is False:
        plt.grid(False)

    #reverse_x = (input("Reverse x-axis? Yes or No")).lower()
    #if reverse_x == 'yes' or reverse_x == 'y':
    #    reverse_x is True
    #    plt.xlim(x_max, x_min)
    #else:
    #    reverse_x is False
    #    plt.xlim(x_min, x_max)

    y_max = set_y_lim(graph_type=graph_type, show_peak_labels=show_peak_labels)

    if title:
        plt.title(str(title).strip(), color = text)
    if random_title:
        plt.title(str(generate_random_title()), color=text)

    ax.set_facecolor(fig_bg)
    plt.xlabel(x_title, color=text)
    plt.ylabel(y_title, color=text)
    plt.xticks(color=text)
    plt.yticks(color=text)
    ax.yaxis.set_major_locator(major_locator)
    plt.ylim(y_min, y_max)

    if save_path:
        try:
            ax.savefig(save_path, facecolor=ax.get_facecolor(), bbox_inches='tight', dpi=dpi)
        except Exception:
            plt.savefig(save_path, facecolor=plt.gcf().get_facecolor(), bbox_inches='tight', dpi=dpi)




def plot(
    data_df,
    detect_columns,
    nm_col,
    int_col,
    force_nist = None,
    graph_type = None,
    scale_mode = 'auto',
    title = None,
    random_title = None,
    show_peak_labels = None,
    show_grid = True,
    show_label_colour = None,
    scale_by_int = None,
    save_path=None,
):
    df_data, has_any_nist, _, _, _ = run_nist_check(data_df=data_df, detect_columns=detect_columns, force_nist=force_nist, nm_col=nm_col, int_col=int_col)
    if has_any_nist is True:
        prep_type = 'nist'
        nist_plot_type = 'g' or 'gaussian' or 't' or 'traditional'
        if nist_plot_type == 'g' or nist_plot_type == 'gaussian':
            graph_type = 'gaussian'
            axis_labels(graph_type=graph_type, show_grid=show_grid, show_peak_labels=show_peak_labels, title=title, random_title=random_title)
            plot_funcs.gaussian_iteration(df_plot_data=df_data, detect_columns=detect_columns,show_peak_labels=show_peak_labels, show_label_colour=show_label_colour, int_col=int_col, nm_col=nm_col)
        if nist_plot_type =='t' or nist_plot_type == 'traditional':
            scale_mode = 'raw'
            graph_type = 'traditional'
            trad_spec(data_df=data_df, detect_columns=detect_columns, scale_mode=scale_mode, show_peak_labels=show_peak_labels, title=title, random_title=random_title)

    else:
        prep_type = 'generic'
        if graph_type == 'traditional':
            trad_spec(data_df=data_df, detect_columns=detect_columns, scale_mode=scale_mode, show_peak_labels=show_peak_labels, title=title, random_title=random_title)

        else:
            data_df, should_exit_early = prep_other(data_df = data_df, detect_columns=detect_columns, int_col=int_col, nm_col=nm_col)
            if should_exit_early:
                fig, ax = plt.subplots(figsize=fig_size, dpi=DPI)
                ax.set_axis_off()
                print("Ending Rendering Early.")
                return fig

            axis_labels(graph_type=graph_type, show_grid=show_grid, show_peak_labels=show_peak_labels, title=title, random_title=random_title)
            get_rgb_type(graph_type=graph_type)
            if rgb_type == 'yes':
                scale_by_int=scale_by_int  # noqa: PLW0127


            if graph_type == 'bar':
                bar_iteration(data_df = data_df, show_peak_labels=show_peak_labels, show_label_colour=show_label_colour, scale_by_int=scale_by_int)
            elif graph_type == 'scatter':
                scatter_iteration(data_df = data_df, show_peak_labels=show_peak_labels, show_label_colour=show_label_colour, scale_by_int=scale_by_int)
            elif graph_type == 'gaussian':
                gaussian_iteration(df_plot_data=data_df, show_peak_labels=show_peak_labels, show_label_colour=show_label_colour, scale_by_int=False)
            elif graph_type == 'line':
                line_plot_iteration(data_df = data_df, show_peak_labels=show_peak_labels, show_label_colour=show_label_colour, scale_by_int=scale_by_int)
            elif graph_type == 'filled line':
                filled_plot(data_df = data_df, show_peak_labels=show_peak_labels, show_label_colour=show_label_colour, scale_by_int=scale_by_int)
            elif graph_type == 'non rgb line':
                non_rgb_iteration(data_df = data_df, show_peak_labels=show_peak_labels, show_label_colour=show_label_colour,)




import matplotlib.patheffects as pe


detection_col = '_raw_int'
int_col = '_adj_int'


# ================
# Traditonal Plot
# ================

# updated to work with trad_spec().

def plot_trad(
    df_plot_data,
    scale_mode,
    detect_columns,
    nm_col,
    has_any_nist,
    #mode,
    show_grid = False,
    scale_by_int = None,
    save_path = None,
    prominence_percentage=0,     
    fig_size=FIG_SIZE,
    min_brightness=0,
    peak_wavelengths=None,
    show_peak_labels=True,
    x_min=X_MIN,
    x_max=X_MAX,
    min_needle_max_width_nm=MIN_NEEDLE_WIDTH,
    max_needle_max_width_nm=MAX_NEEDLE_WIDTH,
    needle_shape_power=NEEDLE_POWER_SHAPE,
    glow_width_multiplier=GLOW_WIDTH_MULT,
    glow_alpha=0,
    dpi=DPI,
    peak_label_y_position=0,
    label_min_norm_int=LABEL_NORM_INT,
    subplots_adjust_top = 0.90,
    max_needle_y = MAX_Y_SCALE,
    fig_height_overflow_scale = fig_height_overflow_scale,
    fig_height_base = FIG_HEIGHT_BASE,
    #title=None,
    #random_title=None,
    #save_path = None,
):

    fig, ax = plt.subplots(figsize=fig_size, dpi=dpi)
    trad_spec_labels(fig=fig, ax=ax, x_min=x_min, x_max=x_max,)

    if has_any_nist:
        scale_mode = None
        min_brightness = DEFAULT_MIN_BRIGHT
        glow_alpha = DEFAULT_GLOW_ALPHA
        prominence_percentage = DEFAULT_PROM_PERC
        peak_label_y_position = DEFAULT_PEAK_LABEL_POSN
    else:
        if scale_mode == 'raw':
            glow_alpha = DEFAULT_GLOW_ALPHA
            peak_label_y_position = DEFAULT_PEAK_LABEL_POSN
        elif scale_mode == 'normalize':
            glow_alpha = NORM_GLOW_ALPHA
            peak_label_y_position = NORM_PEAK_LABEL_POSN

    # --- Peak Detection ---
    if peak_wavelengths is not None:
        nm_vals = df_plot_data[wl_col].values
        peaks = []
        for pw in peak_wavelengths:
            try:
                pv = float(pw)
            except Exception:
                continue
            idx = int(np.argmin(np.abs(nm_vals - pv)))
            peaks.append(idx)
        peaks = sorted(set(peaks))
    elif has_any_nist:
        peaks = nist_codes.identify_spectral_peaks(df_plot_data.reset_index(drop=True), prominence_percentage, peak_wavelengths=None)
    else:
        raw_min = df_plot_data[detection_col].min()
        raw_max = df_plot_data[detection_col].max()
        raw_range = raw_max - raw_min
        if scale_mode == 'raw':
            prominence_percentage = DEFAULT_PROM_PERC
        if scale_mode == 'normalize':
            prominence_percentage = NORM_PROM_PERC
        dynamic_prominence = prominence_percentage * (raw_range if raw_range != 0 else 1.0)
        peaks, _ = find_peaks(df_plot_data[INT_col], prominence=dynamic_prominence)
    
    peak_nms = [float(df_plot_data.iloc[index][wl_col]) for index in peaks]
    peak_ints = [float(df_plot_data.iloc[index]["Norm_Int"]) for index in peaks]
    init_peak_label_y_posn = peak_label_y_position
    try:
        label_ys = compute_label_positions(
            peak_nms,
            intensities=peak_ints,
            base_y=init_peak_label_y_posn,
            min_sep_nm=0.4,
            y_step=0.08,
            method="prefer_stronger_top",
            max_y=0.90,
        )
    except Exception:
        label_ys = [init_peak_label_y_posn] * len(peaks)

    if label_ys:
        max_label_y = max(label_ys)
        overflow = max(0, max_label_y - init_peak_label_y_posn)
        print("\noverflow:", overflow)
        print("o.g. peak label y pos'n:", init_peak_label_y_posn)
        if overflow > 0:
            new_height = fig_height_base + overflow * fig_height_overflow_scale
            new_fig_height = new_height
            fig.set_figheight(new_height)
            current_needle_height_in = max_needle_y * new_height
            new_needle_height = (max_needle_y * fig_height_base) / new_height 
            max_needle_y_scale = new_needle_height
            new_peak_y_posn = (init_peak_label_y_posn * fig_height_base) / new_height
            peak_label_y_position = new_peak_y_posn
            
            print("needle height goal:", new_needle_height, "\n \t inches:", new_needle_height*new_height)
            print("current needle height (in):", current_needle_height_in)
            print("height:", new_height, "max label y:", max_label_y)
            print("\npeak label y pos'n goal:", new_peak_y_posn, "\nnew fig height:", new_fig_height)      
        else:
            print("Labels fit — no expansion needed")
            max_needle_y_scale = max_needle_y
            print("peak label y pos'n:", peak_label_y_position)
    else:
        max_needle_y_scale = max_needle_y
        print("No labels on this spectrum")
    print(f"DEBUG: max_label_y = {max(label_ys) if label_ys else 'N/A'}")

    try:
        label_ys = compute_label_positions(
            peak_nms,
            intensities=peak_ints,
            base_y=peak_label_y_position,   # now uses updated value
            min_sep_nm=1.5,
            y_step=0.08,
            method="prefer_stronger_top",
            max_y=0.85,
        )
    except Exception:
        label_ys = [peak_label_y_position] * len(peaks)
    print("\n new peak label pos'n:", peak_label_y_position)

    # Render every transition as a faint needle (increase visibility for verification)
    _bg_y_top = max_needle_y_scale  # use full height for visibility
    _bg_y = np.linspace(0, _bg_y_top, 40)
    for _idx, _row in df_plot_data.iterrows():
        _nm = float(_row[nm_col])
        _ni = float(_row.get('Norm_Int', 0.0))
        _base_rgb = rgb(_nm)
        _final_scale = final_scale(min_brightness, _ni)
        _color = (_base_rgb[0] * _final_scale, _base_rgb[1] * _final_scale, _base_rgb[2] * _final_scale)
        _width_mult = float(_row.get('_width_mult', 1.0))
        _bg_base_width = min_needle_max_width_nm * 1.0 * _width_mult   # make background lines thicker for test
        _bg_widths = _bg_base_width * np.ones_like(_bg_y)
        _x_left = left_x(_nm, _bg_widths)
        _x_right = right_x(_nm, _bg_widths)
        ax.fill_betweenx(_bg_y, _x_left, _x_right, facecolor=_color, alpha=glow_alpha, edgecolor='none', linewidth=0, zorder=0)

    # Plot sharp distinct lines for each identified peak using fill_betweenx for needle shape
    for j, peak_index in enumerate(peaks):
        peak_nm = df_plot_data.iloc[peak_index][nm_col]
        peak_norm_int = df_plot_data.iloc[peak_index]['Norm_Int']
        base_rgb = rgb(peak_nm)

        # Emphasize peaks: gentle gamma + emphasis multiplier for labeled peaks
        _peak_gamma = 0.8
        if has_any_nist or scale_mode == 'raw':   
            _peak_emphasis = DEFAULT_PEAK_EMPHASIS
            min_brightness = DEFAULT_MIN_BRIGHT
        elif scale_mode == 'normalize':
            _peak_emphasis = NORM_PEAK_EMPHASIS
            min_brightness = NORM_MIN_BRIGHT
        pre_int_scale = min_brightness + (1 - min_brightness) * (peak_norm_int ** _peak_gamma)
        final_int_scale = min(1.0, pre_int_scale * _peak_emphasis)
        color_rgb = colored_rgb(base_rgb, final_int_scale)

        # Scale the maximum width of the needle based on normalized intensity
        base_width = min_needle_max_width_nm + (max_needle_max_width_nm - min_needle_max_width_nm) * peak_norm_int
        width_mult = df_plot_data.iloc[peak_index].get('_width_mult', 1.0)

        # boost peak widths so labeled peaks stand out
        _peak_width_boost = 1.6
        scaled_max_width_nm = base_width * width_mult * _peak_width_boost
        
        # Calculate the actual height the needle should reach (fraction of 0-1)
        current_peak_render_height = max_needle_y_scale

        # Define y-coordinates for the needle shape, spanning from 0 to current_peak_render_height
        y_coords_render = np.linspace(0, current_peak_render_height, 120)
        # Calculate normalized y-coordinates for the width profile
        y_coords_normalized_for_width = (y_coords_render / current_peak_render_height if current_peak_render_height > 0 else np.zeros_like(y_coords_render))

        # Use a power-law profile for the width, making tips slimmer
        width_profile_factor = (4 * y_coords_normalized_for_width * (1 - y_coords_normalized_for_width)) ** needle_shape_power

        # --- Plot the GLOW effect first ---
        glow_current_widths_nm = scaled_max_width_nm * glow_width_multiplier * width_profile_factor
        glow_x_left = left_x(peak_nm, glow_current_widths_nm)
        glow_x_right = right_x(peak_nm, glow_current_widths_nm)
        ax.fill_betweenx(
            y_coords_render,
            glow_x_left,
            glow_x_right,
            facecolor=color_rgb,
            alpha=glow_alpha,
            edgecolor='none',
            linewidth=0,
            antialiased=True,
            zorder=1,
        )

        # --- Plot the main NEEDLE on top of the glow ---
        current_widths_nm = dynamic_prominence(scaled_max_width_nm, width_profile_factor)
        x_left = left_x(peak_nm, current_widths_nm)
        x_right = right_x(peak_nm, current_widths_nm)
        ax.fill_betweenx(
            y_coords_render,
            x_left,
            x_right,
            facecolor=color_rgb,
            edgecolor='none',
            linewidth=0,
            antialiased=True,
            zorder=2,
        )

        # Add text label for the peak wavelength with rotation and stroke for readability
        if show_peak_labels and peak_norm_int >= label_min_norm_int:
            y_for_label = label_ys[j] if j < len(label_ys) else peak_label_y_position
            ax.text(
                peak_nm,
                y_for_label,
                f"{peak_nm:.2f}",
                color="white",
                ha="left",
                va="bottom",
                fontsize=8,
                rotation=60,
                rotation_mode="anchor",
                zorder=3,
                path_effects=[pe.withStroke(linewidth=1.5, foreground="black")],
            )

    if has_any_nist or scale_mode == 'raw':     # scale_mode == 'raw':
        pad = 15
    else:
        pad = 10
    plt.tight_layout(rect=[0, 0, 1, 0.92])


    if scale_mode is not None:
        plt.title(f'{scale_mode.capitalize()} Emission Spectrum Visualization', color=COLOUR, y=0.98, pad=pad)
    else:
        plt.title('Emission Spectrum Visualization', color=COLOUR, y=0.98, pad=pad)
    plt.subplots_adjust(top=subplots_adjust_top)

    print("\nDEBUG: fig height = ", fig.get_figheight(), "\n current needle height = ", current_peak_render_height)


    if save_path:
        try:
            fig.savefig(save_path, facecolor=fig.get_facecolor(), bbox_inches='tight', dpi=dpi)
        except Exception:
            plt.savefig(save_path, facecolor=plt.gcf().get_facecolor(), bbox_inches='tight', dpi=dpi)


    '''
    # Save the plot if a save_path is provided
    if save_path:
        try:
            fig.savefig(save_path, facecolor=fig.get_facecolor(), bbox_inches='tight', dpi=dpi)
        except Exception:
            plt.savefig(save_path, facecolor=plt.gcf().get_facecolor(), bbox_inches='tight', dpi=dpi)
    '''

    return fig, ax


# =========================
# Other Plot Functions
# =========================

def gaussian_iteration(
        df_plot_data, 
        show_peak_labels,
        show_label_colour,
        detect_columns,
        nm_col,
        int_col,
        #mode,
        save_path=None,
        scale_by_int=False,
        scale_mode = None, 
        peak_wavelengths=None,
):
    
    df_plot_data, should_exit_early, has_any_nist = prep_with_nist(data_df = df_plot_data, detect_columns=detect_columns, nm_col=nm_col, int_col=int_col,)
    if should_exit_early:
        fig, ax = plt.subplots(figsize=fig_size, dpi=DPI)
        ax.set_axis_off()
        print("Ending Rendering Early.")
        return fig

    if peak_wavelengths is not None:
        nm_vals = df_plot_data[wl_col].values
        peaks_indices = []
        for pw in peak_wavelengths:
            try:
                pv = float(pw)
            except Exception:
                continue
            idx = int(np.argmin(np.abs(nm_vals - pv)))
            peaks_indices.append(idx)
        peaks_indices = sorted(set(peaks_indices))
    elif has_any_nist:
        peaks_indices = peaks_indices, _properties = find_peaks(df_plot_data['_adj_int'], prominence=DEFAULT_PROM_PERC) 
    else:
        dyn_prominence = dynamic_prominence(DEFAULT_PROM_PERC, int_range)
        peaks_indices, _properties = find_peaks(df_plot_data[INT_col], prominence=dyn_prominence)

    int_vals = df_plot_data['_raw_int'].values
    y_max = int_vals.max()

    # Create a new, denser wavelength array for plotting the synthetic spectrum
    x_synthetic = np.linspace(400, 750, 1000) # 1000 points for a smooth synthetic curve
    y_synthetic = np.zeros_like(x_synthetic) 

    for i, peak_idx in enumerate(peaks_indices):
        peak_nm = float(df_plot_data.iloc[peak_idx][wl_col])
        peak_amplitude = float(df_plot_data.iloc[peak_idx]['_adj_int'])
        normalized_amplitude = float(df_plot_data.iloc[peak_idx]['Norm_Int'])

        # Scale sigma based on normalized intensity (higher intensity = broader peak)
        sigma = base_sigma_nm + (max_sigma_multiplier - 1) * base_sigma_nm * normalized_amplitude

        # Create a Gaussian curve for this peak
        gaussian_curve = peak_amplitude * np.exp(-((x_synthetic - peak_nm)**2) / (2 * sigma**2))
        y_synthetic += gaussian_curve

    # Normalize the synthetic spectrum intensities for coloring (if desired, not strictly necessary for area plot)
    min_y_synthetic = y_synthetic.min()
    max_y_synthetic = y_synthetic.max()
    if (max_y_synthetic - min_y_synthetic) == 0:
        normalized_y_synthetic = np.ones_like(y_synthetic)
    else:
        normalized_y_synthetic = (y_synthetic - min_y_synthetic) / (max_y_synthetic - min_y_synthetic)
    
    # Iterate through each segment of the synthetic spectrum to apply color and alpha
    for i in range(len(x_synthetic) - 1):
        wavelength_start = x_synthetic[i]
        wavelength_end = x_synthetic[i+1]
        base_rgb = rgb(wavelength_start, gamma=gamma_factor)
        alpha = final_scale(min_alpha, normalized_y_synthetic[i])
        plt.fill_between([wavelength_start, wavelength_end],
                        [0, 0],
                        [y_synthetic[i], y_synthetic[i+1]],
                        color=base_rgb,
                        alpha=alpha,
                        linewidth=0)

    if show_peak_labels is True:
        peak_labels(data_df=df_plot_data, show_label_colour=show_label_colour, )

    if save_path:
        try:
            fig.savefig(save_path, facecolor=fig.get_facecolor(), bbox_inches='tight', dpi=DPI)
        except Exception:
            plt.savefig(save_path, facecolor=plt.gcf().get_facecolor(), bbox_inches='tight', dpi=DPI)
            
    print('\n\ny lim = ', plt.ylim())
    print('\nMin y synthetic = ', min_y_synthetic, '\nmax y synthetic = ', max_y_synthetic, '\n y max (from int_vals) = ', y_max, "\nnorm'd y synth = ", normalized_y_synthetic[i])



def line_plot_iteration(data_df, show_peak_labels, show_label_colour, scale_by_int):
    if show_peak_labels is True:
        peak_labels(data_df=data_df, show_label_colour=show_label_colour)

    for i in range(len(data_df) - 1):
        wavelength_start = float(data_df.iloc[i][wl_col])
        wavelength_end = float(data_df.iloc[i+1][wl_col])
        int_factor = float(data_df.iloc[i]['Norm_Int'])
        base_rgb = rgb(wavelength_start)
        if scale_by_int is True:
            final_intensity_scale = final_scale(DEFAULT_MIN_BRIGHT, int_factor)     # this is what does the brightness intensity
        else:
            final_intensity_scale = 1.0
        color_rgb = colored_rgb(base_rgb, final_intensity_scale)       # trying at 1.0, changed from final_intensity_scale. 
        plt.plot([wavelength_start, wavelength_end],
                [data_df.iloc[i][INT_col], data_df.iloc[i+1][INT_col]],
                color=color_rgb,
                linewidth=2)
    
        
def filled_plot(data_df, show_peak_labels, show_label_colour, scale_by_int):
    if show_peak_labels is True:
        peak_labels(data_df=data_df, show_label_colour=show_label_colour)

    for i in range(len(data_df) - 1):
        wavelength_start = data_df.iloc[i][wl_col]
        wavelength_end = data_df.iloc[i+1][wl_col]
        base_rgb = rgb(wavelength_start)
        alpha_factor = data_df.iloc[i]['Norm_Int']  # this is what does the brightness intensity
        if scale_by_int is True:
            alpha = final_scale(min_alpha, alpha_factor)
        else:
            alpha = 1.0    # for NO dimming, alpha has to = 1.0. want this to be optional for filled plots.

        plt.fill_between([wavelength_start, wavelength_end],
                        [0, 0], # Base of the fill is y=0
                        [data_df.iloc[i][INT_col], data_df.iloc[i+1][INT_col]], # Top of the fill
                        color=base_rgb,
                        alpha=alpha,        # alpha has to be rly low to see the top outline. might only be reasonable to do for dimmed plots.
                        linewidth=0) # No line for the fill edges
            
            # Plot the line on top for clarity, using the base color for the line itself
        plt.plot([wavelength_start, wavelength_end],
                    [data_df.iloc[i][INT_col], data_df.iloc[i+1][INT_col]],
                    color=base_rgb,
                    linewidth=2) # Thicker line for better visibility on top of fill        # this should only be for non-smoothed
            
def scatter_iteration(data_df, show_peak_labels, show_label_colour, scale_by_int):
    if show_peak_labels is True:
        peak_labels(data_df=data_df, show_label_colour=show_label_colour)

    alpha_factor = data_df['Norm_Int']
    sizes = base_marker_size + (max_marker_size_factor * alpha_factor)
    if scale_by_int is True:
        alphas = final_scale(min_alpha_scatter, alpha_factor)       # this does the dimming
    else:
        alphas = 1.0
    rgba_colors = []
    for i in range(len(data_df)):
        wavelength = data_df.iloc[i][wl_col]
        r, g, b = rgb(wavelength)
        if scale_by_int is True:
            a = alphas.iloc[i]
        else:
            a = alphas
        rgba_colors.append((r, g, b, a))
    plt.scatter(
        x=data_df[wl_col],
        y=data_df[INT_col],
        c=rgba_colors,
        s=sizes,
        edgecolors='none',
        label='Emission Data'
    )

def bar_iteration(data_df, show_peak_labels, show_label_colour, scale_by_int):
    bar_colors = []
    if show_peak_labels is True:
        peak_labels(data_df=data_df,  show_label_colour=show_label_colour)

    for index, row in data_df.iterrows():
        wavelength = row[wl_col]
        normalized_intensity = row['Norm_Int']

        base_rgb = rgb(wavelength)
        if scale_by_int is True:
            final_intensity_scale = final_scale(DEFAULT_MIN_BRIGHT, normalized_intensity)     # this does the dimming
        else:
            final_intensity_scale = 1.0
        color_rgb = colored_rgb(base_rgb, final_intensity_scale)        # changed to 1.0 from final_intensity_scale. want this to be optional
        bar_colors.append(color_rgb)
    plt.bar(
        x=data_df[wl_col],
        height=data_df[INT_col],
        width=bar_width,
        color=bar_colors,
        edgecolor='none' # No edge color for bars
    )


def non_rgb_iteration(data_df, show_peak_labels, show_label_colour):
    plt.plot(data_df[wl_col], data_df[INT_col], marker=None)
    if show_peak_labels is True:
        peak_labels(data_df=data_df, show_label_colour=show_label_colour)







import math

import string


#show_grid = None

# ===============
# From utils.py
# ===============

def generate_random_title():
    random_code = ''.join(random.choices(string.digits, k=4))
    return f'Spectrum-{random_code}'


# ===========================
# Prompting Functions
# ===========================

def scale_by_int():
    global SCALE_BY_INT
    scale = input("Scale brightness by intensity? Yes or No").lower()
    if scale == 'yes' or scale == 'y':
        SCALE_BY_INT = 'yes'
    elif scale == 'no' or scale == 'n':
        SCALE_BY_INT = 'no'
    print('\nScale brightness:', SCALE_BY_INT.capitalize())

def get_mode():
    global mode
    inp = input("Choose Mode: Dark or Light").lower()
    if inp == 'dark' or inp == 'd':
        mode = 'dark'
    elif inp == 'light' or inp == 'l':
        mode = 'light'
    return mode

def text_colour(mode):
    global colour
    global bg
    if mode == 'dark':
        colour = 'white'
        bg = 'black'
    if mode == 'light':
        colour = 'black'
        bg = 'white'
    return colour, bg




# =======================
# Calculation Functions
# =======================

def dynamic_prominence(prominence, int_range):
    dyn_prom = prominence * int_range
    return dyn_prom

# for final_alpha AND final_intensity_scale
def final_scale(min, factor):
    final = min + (1 - min) * factor
    return final

def left_x(a, b):
    xleft = a - b / 2
    return xleft

def right_x(a, b):
    xright = a + b / 2
    return xright

def round_to_multiple(num, mult):
    rounded = math.ceil(num / mult) * mult 
    return rounded

def colored_rgb(base_rgb, final_intensity_scale):
    colored_rgb = (base_rgb[0] * final_intensity_scale,
                   base_rgb[1] * final_intensity_scale,
                   base_rgb[2] * final_intensity_scale)
    return colored_rgb

def set_y_lim(graph_type, show_peak_labels):
    if graph_type == 'scatter' or show_peak_labels is True:
        if prep_utils.y_max > 230:
            y_max = round_to_multiple(prep_utils.y_max, 100)
        else:
            y_max = round_to_multiple(prep_utils.y_max, 50)
    else:
        y_max = round_to_multiple(prep_utils.y_max, 50)
    print('(set_y_lim)', y_max)
    return y_max



# ==================================
# Other Image Formatiing Functions
# ==================================

def peak_labels(data_df, show_label_colour, ):
    peaks, _ = find_peaks(data_df[INT_col], prominence=dynamic_prominence(DEFAULT_PROM_PERC, int_range)) # Using dynamic prominence
    # Label the identified sharp peaks
    for peak_index in peaks:
        row = data_df.iloc[peak_index]
        wl_labels = float(data_df.iloc[peak_index][wl_col])
        base_rgb = rgb(wl_labels)
        final_intensity_scale = 1.0
        color_rgb = colored_rgb(base_rgb, final_intensity_scale)
        
        if show_label_colour is True: #and mode == 'dark':
            plt.annotate(f"{row[wl_col]:.2f} nm", # Formatted nm to two decimal places
                        (row[wl_col], row[INT_col]),
                        textcoords="offset points", # Offset the text
                        xytext=(0,10), # Distance from point to label
                        ha='center', # Horizontal alignment
                        color=color_rgb,        # this makes the text rgb
                        )
        else:
            plt.annotate(f"{row[wl_col]:.2f} nm", # Formatted nm to two decimal places
                        (row[wl_col], row[INT_col]),
                        textcoords="offset points", # Offset the text
                        xytext=(0,10), # Distance from point to label
                        ha='center', # Horizontal alignment
                        color = COLOUR,
                        )
         



# =============================
# Traditional Spec Codes
# =============================

def get_generic_type():
    global generic_type
    gen_type = (input('Choose Rendering: Raw or Normalised')).lower()
    if gen_type == 'raw' or gen_type == 'r':
        generic_type = 'raw'
    elif gen_type == 'normalised' or gen_type == 'norm' or gen_type == 'n':
        generic_type = 'normalised'
    print ('Rendering Type: ', generic_type)
    return generic_type

def trad_spec_labels(fig, ax, x_min, x_max, figure_bg_color = BG, text_color = COLOUR):

    fig.patch.set_facecolor(figure_bg_color)
    ax.set_facecolor('black')  
    ax.yaxis.set_visible(False)
    ax.set_xlabel(X_TITLE, color=text_color)
    ax.xaxis.set_major_locator(major_locator)
    ax.xaxis.set_minor_locator(minor_locator)
    ax.tick_params(axis='x', which='major', colors=text_color, labelsize=10)
    ax.tick_params(axis='x', which='minor', colors=text_color, length=4, width=0.5)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(0,1)


    #text_colour(mode=mode)

'''
    global figure_bg_color
    global text_color

    if mode == 'dark':
        figure_bg_color = 'black'
        text_color = colour
    elif mode == 'light':
        figure_bg_color = 'white'
        text_color = colour
'''






def _manual_col(data_df, value):
    if value is None or value == "":
        return None
    if isinstance(value, str) and value.isdigit():
        idx = int(value)
        return data_df.columns[idx]
    return value







def resolve_column(df, candidates, label):
    headers = [(str(col).strip(), str(col).strip().lower()) for col in df.columns]

    # Build a flat keyword list from all candidates.
    # Each candidate may be a phrase; we split on non-alphanumeric characters.
    keywords = []
    for candidate in candidates:
        text = str(candidate).strip().lower()
        if not text:
            continue
        parts = [part for part in re.split(r'[^a-z0-9]+', text) if part]
        keywords.extend(parts if parts else [text])

    # Prefer exact matches first, then keyword containment.
    for original, normalized in headers:
        for candidate in candidates:
            candidate_norm = str(candidate).strip().lower()
            if candidate_norm and normalized == candidate_norm:
                return original
    for original, normalized in headers:
        for keyword in keywords:
            if keyword and keyword in normalized:
                return original
    raise KeyError("Could not find a", label, "column. Available columns:", (df.head()))


def res_col_names(data_df, detect_columns, nm_col, int_col):
    global wl_col
    global INT_col

    def normalize_selected_col(value):
        if value is None or value == "":
            return None

        text = str(value).strip()
        if text.isdigit():
            index = int(text)
            if index < 0 or index >= len(data_df.columns):
                return None
            return data_df.columns[index]

        return text

    if detect_columns is True:
        wl_col = normalize_selected_col(nm_col)
        INT_col = normalize_selected_col(int_col)

        if wl_col is None or INT_col is None:
            return data_df, None, None, True

        return data_df, wl_col, INT_col, False

    try:
        wl_col = resolve_column(data_df, lambda_tokens, "wavelength")
        INT_col = resolve_column(data_df, int_tokens, "intensity")
        return data_df, wl_col, INT_col, False
    except KeyError:
        return data_df, None, None, True


def rgb(wavelength, gamma=gamma_factor):
    wavelength = float(wavelength)
    if wavelength >= 380 and wavelength <= 440:
        attenuation = 0.3 + 0.7 * (wavelength - 380) / (440 - 380)
        R = ((-(wavelength - 440) / (440 - 380)) * attenuation) ** gamma
        G = 0.0
        B = (1.0 * attenuation) ** gamma
    elif wavelength >= 440 and wavelength <= 490:
        R = 0.0
        G = ((wavelength - 440) / (490 - 440)) ** gamma
        B = (1.0) ** gamma
    elif wavelength >= 490 and wavelength <= 510:
        R = 0.0
        G = (1.0) ** gamma
        B = ((-(wavelength - 510) / (510 - 490))) ** gamma
    elif wavelength >= 510 and wavelength <= 580:
        R = ((wavelength - 510) / (580 - 510)) ** gamma
        G = (1.0) ** gamma
        B = 0.0
    elif wavelength >= 580 and wavelength <= 645:
        R = (1.0) ** gamma
        G = ((-(wavelength - 645) / (645 - 580))) ** gamma
        B = 0.0
    elif wavelength >= 645 and wavelength <= 750:
        attenuation = 0.3 + 0.7 * (750 - wavelength) / (750 - 645)
        R = (1.0 * attenuation) ** gamma
        G = 0.0
        B = 0.0
    else:
        R = 0.0
        G = 0.0
        B = 0.0
    return (R, G, B)


def compute_label_positions(peak_nms, intensities=None, base_y=None, min_sep_nm=0.5, y_step=0.04, method="prefer_stronger_top", max_y=0.98):
    if not peak_nms:
        return []

    # prepare indices sorted by wavelength
    idx_sorted = sorted(range(len(peak_nms)), key=lambda i: peak_nms[i])
    result = [base_y] * len(peak_nms)

    # build clusters of peaks closer than min_sep_nm
    clusters = []
    cur = [idx_sorted[0]]
    for i in idx_sorted[1:]:
        if abs(peak_nms[i] - peak_nms[cur[-1]]) <= min_sep_nm:
            cur.append(i)
        else:
            clusters.append(cur)
            cur = [i]
    clusters.append(cur)

    for cluster in clusters:
        if len(cluster) == 1:
            result[cluster[0]] = base_y
            continue

        if method == "prefer_stronger_top" and intensities is not None:
            cluster_sorted = sorted(cluster, key=lambda k: -float(intensities[k]))
            n = len(cluster_sorted)
            if n == 1:
                result[cluster_sorted[0]] = base_y
            else:
                step = min(y_step, (max_y - base_y) / (n - 1))
                for pos, idx in enumerate(cluster_sorted):
                    result[idx] = base_y + pos * step

    return result


def trad_spec(
    data_df, 
    #mode,
    scale_mode,
    show_peak_labels,
    title = None,
    random_title = None,
    show_grid = False,
    show_label_colour = None,
    scale_by_int = None,
    fig_size=FIG_SIZE, 
    dpi=DPI,
):
    df_plot_data, should_exit_early, has_any_nist = prep_utils.prep_with_nist(data_df = data_df)
    if should_exit_early:
        fig, ax = plt.subplots(figsize=fig_size, dpi=dpi)
        ax.set_axis_off()
        print("Ending Rendering Early.")
        return fig
    fig, ax = plot_trad(
        df_plot_data = df_plot_data, 
        nm_col = wl_col, 
        has_any_nist=has_any_nist,
        #mode=mode,
        scale_mode=scale_mode,
        show_peak_labels=show_peak_labels,
    )
    return fig




def load_data(file_path):
    """Automatically loads CSV or Excel files into a pandas DataFrame."""
    if not isinstance(file_path, (bytes, bytearray)):
        raise TypeError("load_data expects raw bytes or a bytearray as input.")
    
    if file_path.startswith(b'PK\x03\x04'):
        excel_dict = pd.read_excel(
            io.BytesIO(file_path), 
            sheet_name=None, 
            engine='openpyxl'
        )
        combined_df = pd.concat(excel_dict.values(), ignore_index=True)
        return combined_df

    else:
        try:
            text_content = file_path.decode('utf-8-sig') 
            csv_stream = io.StringIO(text_content)
            dialect = csv.Sniffer().sniff(csv_stream.read(2048))
            csv_stream.seek(0)
            df = pd.read_csv(csv_stream, sep=dialect.delimiter)
            return df
            
        except Exception as e:
            raise ValueError("File content could not be parsed as an XLSX or CSV file.") from e


In [42]:
H_SHEET = "h test.xlsx"
HE_SHEET = "he test.xlsx"
NIST_SHEET = "oxygen nist 2.xlsx"

H_CSV_SHEET = "h test.csv"

H_DF = load_data(H_SHEET)
HE_DF = load_data(HE_SHEET)
NIST_DF = load_data(NIST_SHEET)

H_CSV_DF = load_data(H_CSV_SHEET)

global df  
df = NIST_DF

print(H_DF.head())

TypeError: load_data expects raw bytes or a bytearray as input.